# 10 — MPC目的関数と重み

QとRを「ゲイン」として暗記せず、予測誤差と入力使用量の交換条件として読みます。

**前提**: `09_contact_and_friction_constraints.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
# 背景: MPC目的関数の重みを上流PyMPCと同じリポジトリ配置・環境変数で調べます。
# 目的: ワークスペースとPyMPCの場所を確定し、後続セルの実行条件を再現可能にします。
# OSに依存しないパス演算を行うためPathを読み込む。
from pathlib import Path
# 環境変数の設定にos、モジュール検索パスの設定にsysを使う。
import os, sys

# Notebookを起動した現在位置を絶対パスへ正規化する。
ROOT = Path.cwd().resolve()
# notebook_pympc直下から起動した場合だけリポジトリルートへ移る。
if ROOT.name == "notebook_pympc":
    # 外部実装をROOT基準で参照できるよう親ディレクトリを採用する。
    ROOT = ROOT.parent
# 上流Quadruped-PyMPCの配置先をROOTから組み立てる。
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
# 誤った起動場所のまま実験を進めないよう実装の存在を検証する。
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
# 同名モジュールの取り違えを防ぐため検索パス未登録時だけ処理する。
if str(PYMPC_ROOT) not in sys.path:
    # 現行リポジトリの実装を最優先でimportするため先頭へ追加する。
    sys.path.insert(0, str(PYMPC_ROOT))

# acados生成物の探索基準を未設定時だけ上流同梱ディレクトリへ合わせる。
os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
# 画面のない環境でもMuJoCoを描画できるよう未設定時はEGLを選ぶ。
os.environ.setdefault("MUJOCO_GL", "egl")
# 実験が参照するワークスペースを目視確認できるよう表示する。
print("workspace :", ROOT)
# 上流実装の参照先を目視確認できるよう表示する。
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 有限時間最適化

\[
\min_{\{x_k,u_k\}}
\sum_{k=0}^{N-1}
  (x_k-x_k^{ref})^TQ(x_k-x_k^{ref})
  +(u_k-u_k^{ref})^TR(u_k-u_k^{ref})
+(x_N-x_N^{ref})^TQ_N(x_N-x_N^{ref})
\]

subject to SRBD dynamics、初期状態、摩擦・接触制約。

\(Q\) を増やすとその状態誤差を高価にし、\(R\) を増やすと入力を使うことを高価にします。
単位が違うため、重みの数値の大小だけを横比較してはいけません。

In [2]:
# 背景: 対角Qの各重みは単独ではなく、典型誤差との積 q_i e_i² で目的関数へ寄与します。
# 目的: 高さ・前進速度・rollの典型誤差を使い、単位の異なる状態のstage cost寄与を比較します。
# 角度変換と数値計算にNumPyを使う。
import numpy as np

# set_weight()が作る対角Qの3成分を、典型誤差で切り出して読む。
# stage cost: l_k = (x-x_ref).T @ Q @ (x-x_ref) + (u-u_ref).T @ R @ (u-u_ref)
# 状態名と単位をキーにして、比較に用いる典型誤差を定義する。
errors = {
    # 高さzの追従誤差を0.03 mとして評価する。
    "z [m]": 0.03,
    # world前進速度vxの追従誤差を0.20 m/sとして評価する。
    "vx [m/s]": 0.20,
    # roll誤差5 degをモデルと同じradへ変換して評価する。
    "roll [rad]": np.deg2rad(5),
}
# 各状態に対応する対角重みq_iを上流設定の代表値として定義する。
weights = {"z [m]": 1500, "vx [m/s]": 100, "roll [rad]": 500}
# 状態ごとに同じ式で寄与を計算するため誤差辞書を走査する。
for name, error in errors.items():
    # 対角Qなので、この状態の寄与は交差項なしのq_i*e_i^2になる。
    contribution = weights[name] * error**2
    # 誤差と無次元化前のcost寄与を並べ、数値比較できるよう表示する。
    print(f"{name:12s}: error={error:.4f}, q_i*e_i^2={contribution:.3f}")

# 重み単体ではzの値がrollより大きいという比較前提を検証する。
assert weights["z [m]"] > weights["roll [rad]"]

z [m]       : error=0.0300, q_i*e_i^2=1.350
vx [m/s]    : error=0.2000, q_i*e_i^2=4.000
roll [rad]  : error=0.0873, q_i*e_i^2=3.808


## 現行コードの読み方

`centroidal_nmpc_nominal.py::set_weight(nx,nu)` が対角Q/Rを作り、
`scipy.linalg.block_diag(Q,R)` がstage costのWになります。
`Vx` と `Vu` は `(x,u)` をcost出力 `y` へ並べる選択行列です。

変更前に、対象状態のindex・単位・典型誤差を確認し、
`weight × error²` の寄与を比較します。

In [3]:
# 背景: 単位の異なる状態では代表scaleを使い、q_i=1/scale_i²とすると誤差1 scaleのcostをそろえられます。
# 目的: z・vx・rollの代表誤差から、単位正規化した学習用重みを計算します。
# 正規化した学習用比較: 同じ物理誤差でもscaleで意味が変わる。
# 各状態の代表誤差をm、m/s、radのモデル単位で定義する。
scales = {"z": 0.05, "vx": 0.5, "roll": np.deg2rad(10)}
# 状態ごとに代表誤差から正規化重みを求める。
for key, scale in scales.items():
    # q_i=1/scale_i²を表示し、代表誤差でq_i e_i²=1になることを確認する。
    print(key, "unit normalized weight =", 1/scale**2)

z unit normalized weight = 399.99999999999994
vx unit normalized weight = 4.0
roll unit normalized weight = 32.828063500117445


## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。